# Multi-Model WLT Fail Prediction

Modular prediction framework for ASIC WLT (Wafer Level Test) part failure prediction.  
Uses the data pipeline from `map_WLT_value_to_CQ_slippage` with **swappable ML models**  
(XGBoost, LightGBM, Random Forest, etc.) with Optuna HPO, cost-sensitive optimization,  
and a side-by-side comparison framework.

**Target:** `WLT_fail` — categorical (CP2/CP3/CP4/Pass) based on F3_ACCGMN_SD exceeding HPM WLT limits  
**Features:** CP2 test measurements (room temperature) + slope features (CP2−CP4 drift)

## 1. Data Loading and Feature Engineering

In [ ]:
import importlib
import data_pipeline
importlib.reload(data_pipeline)
from data_pipeline import load_and_prepare, engineer_features, stratified_group_split

# 7 raw WLT parquet files (DPK456_11..17) — same source as map_WLT_value_to_CQ_slippage
_BASE = "G:/DfsDE/LOC/Rt/BST/09_projects/BMI420/External/02_Product_Development/03_System/04_Accel_System_Development/10_ASIC_evaluations/CA/lot1_lot2_lot3_wafermap_tp_V2"
PARQUET_FILES = [
    f"{_BASE}/DPK456_11.parquet",
    f"{_BASE}/DPK456_12.parquet",
    f"{_BASE}/DPK456_13.parquet",
    f"{_BASE}/DPK456_14.parquet",
    f"{_BASE}/DPK456_15.parquet",
    f"{_BASE}/DPK456_16.parquet",
    f"{_BASE}/DPK456_17.parquet",
]
# Use all parquets
TARGET = 'WLT_fail'

pivoted = load_and_prepare(PARQUET_FILES)
data, features_clean = engineer_features(pivoted, target=TARGET)

Loaded cached pivoted data from _pivoted_cache\pivoted_b51b830e9d47.parquet (50147 rows)
Feature reduction: 48 -> 1 (0 near-constant, 47 correlated)


In [3]:
pivoted.columns

Index(['usn', 'T17_62_FW_Combo_Sense_CBIST:F0_ACCALPN_MEAN_CH_X',
       'T17_62_FW_Combo_Sense_CBIST:F0_ACCALPN_MEAN_CH_Y',
       'T17_62_FW_Combo_Sense_CBIST:F0_ACCALPN_MEAN_CH_Z',
       'T17_62_FW_Combo_Sense_CBIST:F0_ACCALPN_SD_CH_X',
       'T17_62_FW_Combo_Sense_CBIST:F0_ACCALPN_SD_CH_Y',
       'T17_62_FW_Combo_Sense_CBIST:F0_ACCALPN_SD_CH_Z',
       'T17_62_FW_Combo_Sense_CBIST:F0_ACCGMN_MEAN_CH_X',
       'T17_62_FW_Combo_Sense_CBIST:F0_ACCGMN_MEAN_CH_Y',
       'T17_62_FW_Combo_Sense_CBIST:F0_ACCGMN_MEAN_CH_Z',
       'T17_62_FW_Combo_Sense_CBIST:F0_ACCGMN_SD_CH_X',
       'T17_62_FW_Combo_Sense_CBIST:F0_ACCGMN_SD_CH_Y',
       'T17_62_FW_Combo_Sense_CBIST:F0_ACCGMN_SD_CH_Z',
       'T17_62_FW_Combo_Sense_CBIST:F3_ACCALPN_MEAN_CH_X',
       'T17_62_FW_Combo_Sense_CBIST:F3_ACCALPN_MEAN_CH_Y',
       'T17_62_FW_Combo_Sense_CBIST:F3_ACCALPN_MEAN_CH_Z',
       'T17_62_FW_Combo_Sense_CBIST:F3_ACCALPN_SD_CH_X',
       'T17_62_FW_Combo_Sense_CBIST:F3_ACCALPN_SD_CH_Y',
       'T17_

In [4]:
data.WLT_fail.value_counts()

WLT_fail
Pass    48733
CP2      1366
CP4        48
Name: count, dtype: int64

## 2. Model Registry

Each entry defines:
- `model_class`: the sklearn-compatible classifier
- `search_space`: function returning Optuna param suggestions
- `fit_kwargs`: extra kwargs for `.fit()` (e.g. early stopping)
- `supports_sample_weight`: whether the model accepts `sample_weight` in fit

To add a new model, simply register it in `MODEL_REGISTRY`.

In [14]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, accuracy_score, confusion_matrix,
                              f1_score, precision_recall_curve, auc)
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
import numpy as np
import json
import os
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# --- New imports for CatBoost and TabNet ---
try:
    from catboost import CatBoostClassifier
except ImportError:
    CatBoostClassifier = None
try:
    from pytorch_tabnet.tab_model import TabNetClassifier
except ImportError:
    TabNetClassifier = None

# --- Search space definitions ---
def xgb_search_space(trial):
    return {
        'max_depth': trial.suggest_int('max_depth', 2, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }

def lgbm_search_space(trial):
    return {
        'max_depth': trial.suggest_int('max_depth', 2, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'num_leaves': trial.suggest_int('num_leaves', 8, 128),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }

def rf_search_space(trial):
    return {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_float('max_features', 0.1, 1.0),
    }

# --- CatBoost search space ---
def catboost_search_space(trial):
    return {
        'iterations': trial.suggest_int('iterations', 50, 500),
        'depth': trial.suggest_int('depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 1),
    }

# --- TabNet search space ---
def tabnet_search_space(trial):
    return {
        'n_d': trial.suggest_int('n_d', 8, 64),
        'n_a': trial.suggest_int('n_a', 8, 64),
        'n_steps': trial.suggest_int('n_steps', 3, 10),
        'gamma': trial.suggest_float('gamma', 1.0, 2.0),
        'lambda_sparse': trial.suggest_float('lambda_sparse', 1e-6, 1e-3, log=True),
        'momentum': trial.suggest_float('momentum', 0.01, 0.4),
        'clip_value': trial.suggest_float('clip_value', 1.0, 2.0),
    }

# --- Model Registry ---
MODEL_REGISTRY = {
    'xgboost': {
        'model_class': XGBClassifier,
        'search_space': xgb_search_space,
        'fixed_params': {'random_state': 42, 'eval_metric': 'mlogloss',
                         'objective': 'multi:softprob'},
        'fit_kwargs': lambda X_val, y_val: {'eval_set': [(X_val, y_val)], 'verbose': False},
        'supports_early_stopping': True,
        'early_stopping_rounds': 20,
        'supports_sample_weight': True,
    },
    'lightgbm': {
        'model_class': LGBMClassifier,
        'search_space': lgbm_search_space,
        'fixed_params': {'random_state': 42, 'verbosity': -1,
                         'objective': 'multiclass'},
        'fit_kwargs': lambda X_val, y_val: {'eval_set': [(X_val, y_val)]},
        'supports_early_stopping': True,
        'early_stopping_rounds': 20,
        'supports_sample_weight': True,
    },
    'random_forest': {
        'model_class': RandomForestClassifier,
        'search_space': rf_search_space,
        'fixed_params': {'random_state': 42, 'n_jobs': -1},
        'fit_kwargs': lambda X_val, y_val: {},
        'supports_early_stopping': False,
        'early_stopping_rounds': None,
        'supports_sample_weight': True,
    },
    'catboost': {
        'model_class': CatBoostClassifier if CatBoostClassifier else None,
        'search_space': catboost_search_space,
        'fixed_params': {'random_state': 42, 'loss_function': 'MultiClass', 'verbose': False},
        'fit_kwargs': lambda X_val, y_val: {'eval_set': (X_val, y_val)},
        'supports_early_stopping': True,
        'early_stopping_rounds': 20,
        'supports_sample_weight': True,
    },
    'tabnet': {
        'model_class': TabNetClassifier if TabNetClassifier else None,
        'search_space': tabnet_search_space,
        'fixed_params': {'seed': 42, 'verbose': 0},
        'fit_kwargs': lambda X_val, y_val: {'eval_set': [(X_val, y_val)], 'patience': 20},
        'supports_early_stopping': True,
        'early_stopping_rounds': None,
        'supports_sample_weight': False,
    },
}

# --- Select which models to run ---
ACTIVE_MODELS = ['xgboost', 'catboost', 'tabnet']

## 3. Stratified Group Split

In [15]:
split = stratified_group_split(data, target=TARGET)

train_idx = split['train_idx']
val_idx = split['val_idx']
holdout_idx = split['holdout_idx']

X = data[features_clean].values

# Encode target
le = LabelEncoder()
y_all = le.fit_transform(data[TARGET].values)
class_names = list(le.classes_)
n_classes = len(class_names)
pass_idx = class_names.index('Pass')

print(f"\nFeature matrix shape: {X.shape}")
print(f"Train: {len(train_idx)} | Val: {len(val_idx)} | Holdout: {len(holdout_idx)}")
print(f"Classes: {class_names} (Pass index: {pass_idx})")
print(f"Distribution: {dict(zip(class_names, np.bincount(y_all)))}")


7 groups (lots) found: ['DPK456-11', 'DPK456-12', 'DPK456-13', 'DPK456-14', 'DPK456-15', 'DPK456-16', 'DPK456-17']

Per-lot fail rates:
  DPK456-11 (7149 dies): CP2: 0.030 | CP4: 0.001
  DPK456-12 (7197 dies): CP2: 0.029 | CP4: 0.001
  DPK456-13 (7134 dies): CP2: 0.023 | CP4: 0.001
  DPK456-14 (7180 dies): CP2: 0.030 | CP4: 0.001
  DPK456-15 (7131 dies): CP2: 0.027 | CP4: 0.001
  DPK456-16 (7202 dies): CP2: 0.026 | CP4: 0.001
  DPK456-17 (7154 dies): CP2: 0.026 | CP4: 0.002
  Auto n_train_range: (3, 6) (from 7 lots)

Stratified split:
  Train:   21505 dies / 3 wafers ['DPK456-11', 'DPK456-16', 'DPK456-17']
  Val:     7131 dies / 1 wafers ['DPK456-15']
  Holdout: 21511 dies / 3 wafers ['DPK456-12', 'DPK456-13', 'DPK456-14']
  CP2 rate — Train: 0.0272 | Val: 0.0272 | Holdout: 0.0273
  CP4 rate — Train: 0.0012 | Val: 0.0010 | Holdout: 0.0007

Feature matrix shape: (50147, 1)
Train: 21505 | Val: 7131 | Holdout: 21511
Classes: ['CP2', 'CP4', 'Pass'] (Pass index: 2)
Distribution: {'CP2': 13

## 4. Optuna Hyperparameter Optimization Factory

Generic objective factory that works with any sklearn-compatible model from the registry.

In [16]:
# --- Business cost parameters ---
FN_COST = 10   # cost of missing a fail (predicting Pass when part actually fails)
FP_COST = 1    # cost of false alarm (predicting fail when part actually passes)
N_OPTUNA_TRIALS = 50
RUN_OPTUNA = True  # Set True to run HPO; False to use saved params only
PARAMS_DIR = 'optuna_params_multi'
os.makedirs(PARAMS_DIR, exist_ok=True)


def compute_business_cost(y_true, y_pred, pass_idx):
    """Compute business cost: FN = fails predicted as Pass, FP = Pass predicted as fail."""
    cm_v = confusion_matrix(y_true, y_pred, labels=list(range(n_classes)))
    fn = cm_v[:, pass_idx].sum() - cm_v[pass_idx, pass_idx]  # fail rows predicted Pass
    fp = cm_v[pass_idx, :].sum() - cm_v[pass_idx, pass_idx]  # Pass rows predicted fail
    return FN_COST * fn + FP_COST * fp, int(fn), int(fp)


def make_objective(model_name, X_tr, y_tr, X_v, y_v, sw):
    """Generic Optuna objective factory for multi-class classification."""
    cfg = MODEL_REGISTRY[model_name]
    model_class = cfg['model_class']
    search_space_fn = cfg['search_space']
    fixed_params = cfg['fixed_params']
    fit_kwargs_fn = cfg['fit_kwargs']
    supports_es = cfg['supports_early_stopping']
    es_rounds = cfg['early_stopping_rounds']
    supports_sw = cfg['supports_sample_weight']

    def objective(trial):
        params = search_space_fn(trial)
        params.update(fixed_params)
        if supports_es and es_rounds:
            params['early_stopping_rounds'] = es_rounds

        model = model_class(**params)
        fit_kw = fit_kwargs_fn(X_v, y_v)
        if supports_sw:
            fit_kw['sample_weight'] = sw
        model.fit(X_tr, y_tr, **fit_kw)

        y_pred = model.predict(X_v)
        cost, _, _ = compute_business_cost(y_v, y_pred, pass_idx)
        return cost

    return objective


def get_params_file(model_name):
    return os.path.join(PARAMS_DIR, f'{model_name}_best_params.json')


def load_saved_params(model_name):
    path = get_params_file(model_name)
    if os.path.exists(path):
        with open(path, 'r') as f:
            return json.load(f)
    return {}


def save_params(model_name, params_dict):
    path = get_params_file(model_name)
    with open(path, 'w') as f:
        json.dump(params_dict, f, indent=2)

print("Objective factory ready.")
print(f"Cost function: FN×{FN_COST} + FP×{FP_COST} (multi-class: {n_classes} classes)")
print(f"Optuna trials per model: {N_OPTUNA_TRIALS}")
print(f"RUN_OPTUNA: {RUN_OPTUNA} {'(will run HPO)' if RUN_OPTUNA else '(using saved params only)'}")

Objective factory ready.
Cost function: FN×10 + FP×1 (multi-class: 3 classes)
Optuna trials per model: 50
RUN_OPTUNA: True (will run HPO)


## 5. Training Loop (Multi-Class)

For each active model:
1. Load or run Optuna HPO (minimize business cost on validation set)
2. Train final model with best params + sample weight balancing
3. Evaluate on holdout set

In [17]:
X_train, y_train = X[train_idx], y_all[train_idx]
X_val, y_val = X[val_idx], y_all[val_idx]
X_holdout, y_holdout = X[holdout_idx], y_all[holdout_idx]
sample_weights = compute_sample_weight('balanced', y_train)

all_results = {}  # keyed by model_name

for model_name in ACTIVE_MODELS:
    cfg = MODEL_REGISTRY[model_name]
    saved_params = load_saved_params(model_name)

    print(f"\n{'#'*70}")
    print(f"# MODEL: {model_name.upper()}")
    print(f"{'#'*70}")

    # --- HPO ---
    if TARGET in saved_params:
        bp = saved_params[TARGET]
        print(f"  Using saved params: {bp}")
    elif RUN_OPTUNA:
        study = optuna.create_study(direction='minimize',
                                    sampler=optuna.samplers.TPESampler(seed=42))
        study.optimize(
            make_objective(model_name, X_train, y_train, X_val, y_val, sample_weights),
            n_trials=N_OPTUNA_TRIALS
        )
        bp = study.best_params
        print(f"  Optuna best val cost: {study.best_value}")
        print(f"  Best params: {bp}")
    else:
        print(f"  ⚠ No saved params and RUN_OPTUNA=False — using defaults")
        bp = {}

    # Save params (only if we have tuned params)
    if bp:
        save_params(model_name, {TARGET: bp})

    # --- Train final model ---
    train_params = {**bp, **cfg['fixed_params']}
    if cfg['supports_early_stopping'] and cfg['early_stopping_rounds']:
        train_params['early_stopping_rounds'] = cfg['early_stopping_rounds']

    model = cfg['model_class'](**train_params)
    fit_kw = cfg['fit_kwargs'](X_val, y_val)
    if cfg['supports_sample_weight']:
        fit_kw['sample_weight'] = sample_weights
    model.fit(X_train, y_train, **fit_kw)

    best_iter = getattr(model, 'best_iteration', bp.get('n_estimators', None))
    print(f"  Best iteration: {best_iter}")

    # --- Holdout evaluation ---
    y_hold_pred = model.predict(X_holdout)
    y_hold_proba = model.predict_proba(X_holdout)

    acc = accuracy_score(y_holdout, y_hold_pred)
    f1_macro = f1_score(y_holdout, y_hold_pred, average='macro')
    f1_weighted = f1_score(y_holdout, y_hold_pred, average='weighted')
    cm = confusion_matrix(y_holdout, y_hold_pred, labels=list(range(n_classes)))
    report = classification_report(y_holdout, y_hold_pred, labels=list(range(n_classes)),
                                   target_names=class_names, output_dict=True)

    holdout_cost, hold_fn, hold_fp = compute_business_cost(y_holdout, y_hold_pred, pass_idx)

    # Per-class PR-AUC (one-vs-rest for fail classes)
    pr_aucs = {}
    for i, cls in enumerate(class_names):
        if cls == 'Pass':
            continue
        y_bin = (y_holdout == i).astype(int)
        proba_i = y_hold_proba[:, i]
        if y_bin.sum() > 0:
            prec, rec, _ = precision_recall_curve(y_bin, proba_i)
            pr_aucs[cls] = round(auc(rec, prec), 4)

    print(f"\n  Holdout: Acc={acc:.4f} F1M={f1_macro:.4f}")
    print(f"  Business Cost={holdout_cost} (FN={hold_fn}×{FN_COST} + FP={hold_fp}×{FP_COST})")
    print(f"  Per-class PR-AUC: {pr_aucs}")

    # Feature importances
    feat_imp = getattr(model, 'feature_importances_', np.zeros(len(features_clean)))

    all_results[model_name] = {
        'model': model, 'le': le, 'accuracy': acc, 'f1_macro': f1_macro,
        'f1_weighted': f1_weighted, 'confusion_matrix': cm,
        'class_names': class_names, 'report': report,
        'feature_importances': feat_imp,
        'pr_aucs': pr_aucs, 'holdout_cost': int(holdout_cost),
        'fn_cost': FN_COST, 'fp_cost': FP_COST,
        'holdout_fn': hold_fn, 'holdout_fp': hold_fp,
        'features_clean': features_clean,
        'best_iteration': best_iter, 'best_params': bp,
    }

print(f"\n{'='*70}")
print(f"All models trained. Results for {len(all_results)} models.")


######################################################################
# MODEL: XGBOOST
######################################################################
  Using saved params: {'max_depth': 6, 'learning_rate': 0.2359614469254991, 'n_estimators': 80, 'min_child_weight': 16, 'subsample': 0.8848163080975646, 'colsample_bytree': 0.5166518765818108, 'reg_alpha': 2.8095311717855503e-06, 'reg_lambda': 0.00011921087942045072}
  Best iteration: 73

  Holdout: Acc=0.5513 F1M=0.2618
  Business Cost=11761 (FN=243×10 + FP=9331×1)
  Per-class PR-AUC: {'CP2': 0.1054, 'CP4': 0.0007}

######################################################################
# MODEL: CATBOOST
######################################################################
  Using saved params: {'iterations': 261, 'depth': 3, 'learning_rate': 0.2307785791981754, 'l2_leaf_reg': 5.522573682052973, 'bagging_temperature': 0.461441105648478}
  Best iteration: None

  Holdout: Acc=0.7266 F1M=0.3204
  Business Cost=8675 (FN=317×10 + F

c:\Users\MMO6RT\.conda\envs\venv1\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
c:\Users\MMO6RT\.conda\envs\venv1\Lib\site-packages\torch\utils\data\_utils\collate.py:285: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:209.)
  return collate([torch.as_tensor(b) for b in batch], collate_fn_map=collate_fn_map)



Early stopping occurred at epoch 20 with best_epoch = 0 and best_val_0_accuracy = 0.97237


c:\Users\MMO6RT\.conda\envs\venv1\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 20 with best_epoch = 0 and best_val_0_accuracy = 0.97181


...

c:\Users\MMO6RT\.conda\envs\venv1\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


  Optuna best val cost: 1960.0
  Best params: {'n_d': 63, 'n_a': 35, 'n_steps': 4, 'gamma': 1.4316707754336455, 'lambda_sparse': 5.6192902093603254e-05, 'momentum': 0.30304091042584247, 'clip_value': 1.6882295119501993}

Early stopping occurred at epoch 20 with best_epoch = 0 and best_val_0_accuracy = 0.97251


c:\Users\MMO6RT\.conda\envs\venv1\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


  Best iteration: None

  Holdout: Acc=0.9730 F1M=0.3539
  Business Cost=5800 (FN=580×10 + FP=0×1)
  Per-class PR-AUC: {'CP2': 0.132, 'CP4': 0.0009}

All models trained. Results for 3 models.


c:\Users\MMO6RT\.conda\envs\venv1\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\MMO6RT\.conda\envs\venv1\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\MMO6RT\.conda\envs\venv1\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


## 6. Results Summary

Evaluation is performed inline per model. Below: comparison table and visualizations.

## 7. Model Comparison Summary

In [18]:
import pandas as pd

comparison_rows = []
for model_name, r in all_results.items():
    row = {
        'Model': model_name,
        'Holdout Cost': r['holdout_cost'],
        'F1 Macro': round(r['f1_macro'], 4),
        'F1 Weighted': round(r['f1_weighted'], 4),
        'Accuracy': round(r['accuracy'], 4),
        'FN': r['holdout_fn'],
        'FP': r['holdout_fp'],
    }
    # Add per-class PR-AUC
    for cls, val in r['pr_aucs'].items():
        row[f'PR-AUC ({cls})'] = val
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows).sort_values('Holdout Cost')
print("\n" + "="*80)
print("MODEL COMPARISON (sorted by Holdout Business Cost)")
print("="*80)
display(comparison_df)

best_model = comparison_df.iloc[0]['Model']
print(f"\nBest model (lowest cost): {best_model} "
      f"(cost={comparison_df.iloc[0]['Holdout Cost']}, F1M={comparison_df.iloc[0]['F1 Macro']})")


MODEL COMPARISON (sorted by Holdout Business Cost)


,Model,Holdout Cost,F1 Macro,F1 Weighted,Accuracy,FN,FP,PR-AUC (CP2),PR-AUC (CP4)
2,tabnet,5800,0.3539,0.9607,0.9730,580,0,0.1320,0.0009
1,catboost,8675,0.3204,0.8207,0.7266,317,5505,0.1267,0.0008
0,xgboost,11761,0.2618,0.6897,0.5513,243,9331,0.1054,0.0007



Best model (lowest cost): tabnet (cost=5800, F1M=0.3539)


In [19]:
import plotly.express as px
import plotly.graph_objects as go

# --- Business cost bar chart ---
fig = px.bar(
    comparison_df,
    x='Model',
    y='Holdout Cost',
    title='Model Comparison: Holdout Business Cost (lower is better)',
    template='plotly_white',
    text='Holdout Cost',
    color='Model',
)
fig.update_traces(textposition='outside')
fig.update_layout(yaxis_title='Business Cost (FN×10 + FP×1)', showlegend=False)
fig.show()

# --- F1 Macro bar chart ---
fig2 = px.bar(
    comparison_df,
    x='Model',
    y='F1 Macro',
    title='Model Comparison: F1 Macro (higher is better)',
    template='plotly_white',
    text='F1 Macro',
    color='Model',
)
fig2.update_traces(textposition='outside')
fig2.update_layout(yaxis=dict(range=[0, 1.1]), showlegend=False)
fig2.show()

# --- Confusion matrices for all models ---
for model_name, r in all_results.items():
    fig_cm = go.Figure(data=go.Heatmap(
        z=r['confusion_matrix'], x=class_names, y=class_names,
        colorscale='Blues', text=r['confusion_matrix'], texttemplate='%{text}',
    ))
    fig_cm.update_layout(
        title=f'Confusion Matrix — {model_name}',
        xaxis_title='Predicted', yaxis_title='Actual',
        template='plotly_white', height=400, yaxis=dict(autorange='reversed'),
    )
    fig_cm.show()

## 8. Best Model Details

Feature importances and classification report for the best-performing model.

In [20]:
# --- Feature importance for best model ---
best_r = all_results[best_model]
imp = best_r['feature_importances']
top_idx = np.argsort(imp)[-20:]
fi_df = pd.DataFrame({
    'Feature': [features_clean[i] for i in top_idx],
    'Importance': imp[top_idx],
}).sort_values('Importance')

fig_fi = px.bar(fi_df, x='Importance', y='Feature', orientation='h',
                color='Importance', color_continuous_scale='Viridis',
                title=f'Top 20 Feature Importances — {best_model}')
fig_fi.update_layout(template='plotly_white', height=500, showlegend=False)
fig_fi.show()

# --- Classification report for best model ---
print(f"\nClassification report ({best_model}):")
print(classification_report(y_holdout, all_results[best_model]['model'].predict(X_holdout),
                            labels=list(range(n_classes)), target_names=class_names))


Classification report (tabnet):
              precision    recall  f1-score   support

         CP2       1.00      0.04      0.08       587
         CP4       0.00      0.00      0.00        16
        Pass       0.97      1.00      0.99     20908

    accuracy                           0.97     21511
   macro avg       0.66      0.35      0.35     21511
weighted avg       0.97      0.97      0.96     21511



c:\Users\MMO6RT\.conda\envs\venv1\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

c:\Users\MMO6RT\.conda\envs\venv1\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

c:\Users\MMO6RT\.conda\envs\venv1\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

